In [45]:
import numpy as np

y = np.load("Data Proecessing/y_labels.npy")

unique, counts = np.unique(y, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count} beats")

Class 0: 90600 beats
Class 1: 2781 beats
Class 2: 7235 beats
Class 3: 802 beats


In [38]:
import numpy as np

y = np.load("Data Proecessing/y_labels.npy")   # shape (N,)

# label_map: 0=N, 1=S, 2=V, 3=F
idx_N = np.where(y == 0)[0]   # all indices of Normal beats
idx_S = np.where(y == 1)[0]   # all indices of Supraventricular beats
idx_V = np.where(y == 2)[0]   # all indices of Ventricular beats
idx_F = np.where(y == 3)[0]   # all indices of Fusion beats

print("N indices (first 10):", idx_N[:10])
print("S indices (first 10):", idx_S[:10])
print("V indices (first 10):", idx_V[:10])
print("F indices (first 10):", idx_F[:10])

N indices (first 10): [ 0  1  2  3  4  5  6  8  9 10]
S indices (first 10): [   7  230  258  342  441  599  987 1078 1085 1103]
V indices (first 10): [1906 4152 4234 4235 4236 6361 6443 6507 6516 6534]
F indices (first 10): [11609 12165 12929 12951 22230 22322 22402 22588 40046 40065]


In [43]:
import numpy as np
import joblib
import pywt

# Load everything correctly
model = joblib.load("Data Proecessing/rf_model.pkl")
scaler = joblib.load("Data Proecessing/scaler.pkl")
X_beats = np.load("Data Proecessing/X_beats.npy")
y = np.load("Data Proecessing/y_labels.npy")

label_map = {0:'N', 1:'S', 2:'V', 3:'F'}

# Wavelet feature extraction
def extract_features(beat):
    coeffs = pywt.wavedec(beat, 'db4', level=4)
    feat = []
    for c in coeffs:
        p = np.abs(c) / np.sum(np.abs(c))
        feat += [
            np.sum(c**2),
            -np.sum(p*np.log2(p+1e-12)),
            np.mean(c),
            np.var(c),
            np.std(c),
            np.sqrt(np.mean(c**2))
        ]
    return np.array(feat)

# Test a beat from raw ECG
index = 11609   # choose ANY index you want

beat = X_beats[index]
true = y[index]

f = extract_features(beat).reshape(1,-1)
f_s = scaler.transform(f)

pred = model.predict(f_s)[0]




In [44]:
print("Predicted:", label_map[pred])
print("Actual   :", label_map[true])

Predicted: F
Actual   : F
